# 03 — ممیزی دیتاست مستقل UrbanEV برای اعتبارسنجی خارجی

## هدف نوت‌بوک

این نوت‌بوک دیتاست مستقل UrbanEV مربوط به شنژن را بدون هیچ‌گونه بازآموزی، تنظیم مدل یا ادغام با CHARGED بررسی می‌کند.

هدف، شناسایی متغیرهای مشترک و هم‌معنا میان UrbanEV و CHARGED است. فقط ویژگی‌هایی که برای هر دو دیتاست و در ادامه برای شهر اصفهان قابل استخراج یا برآورد باشند، وارد مجموعهٔ ویژگی انتقال‌پذیر نهایی خواهند شد.

این نوت‌بوک فقط ممیزی کیفیت، ساختار، مقیاس زمانی، شناسه‌ها، مختصات و هم‌ارزی مفهومی متغیرها را انجام می‌دهد.

In [1]:
# Prompt: Import required libraries and verify the downloaded UrbanEV external-validation files.

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

candidate_roots = [Path.cwd().resolve(), Path.cwd().resolve().parent]

PROJECT_ROOT = next(
    (
        path
        for path in candidate_roots
        if (path / "data" / "raw" / "urbanev" / "inf.csv").exists()
    ),
    None,
)

assert PROJECT_ROOT is not None, (
    "inf.csv پیدا نشد. بررسی کن فایل‌ها در مسیر "
    "data/raw/urbanev/ قرار داشته باشند."
)

URBANEV_RAW_DIR = PROJECT_ROOT / "data" / "raw" / "urbanev"
OUTPUT_TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"
OUTPUT_FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"

for directory in [OUTPUT_TABLES_DIR, OUTPUT_FIGURES_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

URBANEV_REQUIRED_FILES = [
    "inf.csv",
    "occupancy.csv",
    "duration.csv",
    "poi.csv",
    "weather_airport.csv",
    "weather_central.csv",
    "weather_header.txt",
]

urbanev_file_inventory = pd.DataFrame(
    {
        "file_name": URBANEV_REQUIRED_FILES,
        "exists": [
            (URBANEV_RAW_DIR / file_name).exists()
            for file_name in URBANEV_REQUIRED_FILES
        ],
        "size_mb": [
            (
                (URBANEV_RAW_DIR / file_name).stat().st_size
                / (1024 ** 2)
            )
            if (URBANEV_RAW_DIR / file_name).exists()
            else np.nan
            for file_name in URBANEV_REQUIRED_FILES
        ],
    }
)

assert urbanev_file_inventory["exists"].all(), (
    "حداقل یکی از فایل‌های موردنیاز UrbanEV در پوشه وجود ندارد."
)

display(urbanev_file_inventory)

print(f"Project root: {PROJECT_ROOT}")
print(f"UrbanEV raw-data directory: {URBANEV_RAW_DIR}")

,file_name,exists,size_mb
0,inf.csv,True,0.0710
1,occupancy.csv,True,5.2583
2,duration.csv,True,16.9777
3,poi.csv,True,31.1341
4,weather_airport.csv,True,0.1582
5,weather_central.csv,True,0.1729
6,weather_header.txt,True,0.0005


Project root: F:\UrbanEV_Charging_Demand
UrbanEV raw-data directory: F:\UrbanEV_Charging_Demand\data\raw\urbanev


## ۳-۱. بررسی ساختار فایل‌های اصلی UrbanEV

در این مرحله، ساختار فایل‌های ایستگاه‌ها، تقاضای شارژ و اشغال شارژر بررسی می‌شود. هدف، تعیین دقیق شناسهٔ مکانی، مقیاس زمانی، متغیرهای هدف و تفاوت‌های احتمالی با CHARGED است.

هیچ داده‌ای از UrbanEV برای آموزش یا تنظیم مدل استفاده نخواهد شد.

In [2]:
# Prompt: Inspect UrbanEV schemas and compact samples without displaying hundreds of demand columns.

URBANEV_CSV_FILES = [
    "inf.csv",
    "duration.csv",
    "occupancy.csv",
    "poi.csv",
    "weather_airport.csv",
    "weather_central.csv",
]

urbanev_samples = {}
urbanev_schema_rows = []

for file_name in URBANEV_CSV_FILES:
    sample_df = pd.read_csv(
        URBANEV_RAW_DIR / file_name,
        nrows=5,
    )

    urbanev_samples[file_name] = sample_df.copy()

    urbanev_schema_rows.append(
        {
            "file_name": file_name,
            "sample_rows_read": len(sample_df),
            "column_count": len(sample_df.columns),
            "first_column": sample_df.columns[0],
            "first_10_columns": " | ".join(
                sample_df.columns[:10].astype(str)
            ),
        }
    )

urbanev_schema = pd.DataFrame(urbanev_schema_rows)

display(urbanev_schema)

print("Sample: inf.csv")
display(urbanev_samples["inf.csv"])

print("Sample: duration.csv — first eight columns only")
display(urbanev_samples["duration.csv"].iloc[:, :8])

print("Sample: occupancy.csv — first eight columns only")
display(urbanev_samples["occupancy.csv"].iloc[:, :8])

print("Sample: poi.csv — first eight columns only")
display(urbanev_samples["poi.csv"].iloc[:, :8])

weather_header_text = (
    URBANEV_RAW_DIR / "weather_header.txt"
).read_text(encoding="utf-8")

print("Weather-header documentation:")
print(weather_header_text[:3000])

urbanev_schema.to_csv(
    OUTPUT_TABLES_DIR / "urbanev_schema_audit.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Saved: outputs/tables/urbanev_schema_audit.csv")

,file_name,sample_rows_read,column_count,first_column,first_10_columns
0,inf.csv,5,7,station_id,station_id | longitude | latitude | charge_cou...
1,duration.csv,5,276,time,time | 102 | 104 | 105 | 106 | 107 | 108 | 109...
2,occupancy.csv,5,276,time,time | 102 | 104 | 105 | 106 | 107 | 108 | 109...
3,poi.csv,5,3,primary_types,primary_types | longitude | latitude
4,weather_airport.csv,5,7,time,time | T | P0 | P | U | nRAIN | Td
5,weather_central.csv,5,7,time,time | T | P0 | P | U | nRAIN | Td


Sample: inf.csv


,station_id,longitude,latitude,charge_count,TAZID,area,perimeter
0,1001,113.7847,22.7141,20,559,"5,726,523.0560","9,624.5755"
1,1002,113.7850,22.7259,22,558,"3,927,542.0450","8,189.3687"
2,1003,113.7880,22.7355,6,558,"3,927,542.0450","8,189.3687"
3,1004,113.7881,22.6934,11,596,"2,972,113.2410","7,335.1002"
4,1006,113.7910,22.7314,10,594,"2,044,150.5270","5,825.1986"


Sample: duration.csv — first eight columns only


,time,102,104,105,106,107,108,109
0,2022-09-01 00:00:00,8.8333,1.5833,9.1667,11.2500,13.1667,11.2500,8.4167
1,2022-09-01 01:00:00,8.1667,1.3333,11.0000,11.6667,14.4167,12.5000,7.4167
2,2022-09-01 02:00:00,8.9167,1.9167,11.0000,12.7500,13.6667,12.1667,8.3333
3,2022-09-01 03:00:00,9.1667,1.8333,10.0000,10.5833,13.0833,12.5833,8.9167
4,2022-09-01 04:00:00,9.2500,1.7500,10.4167,11.1667,12.8333,12.0833,8.0833


Sample: occupancy.csv — first eight columns only


,time,102,104,105,106,107,108,109
0,2022-09-01 00:00:00,17.0000,1.0000,15.0000,54.0000,23.0000,13.0000,11.0000
1,2022-09-01 01:00:00,16.0000,1.0000,19.0000,50.0000,25.0000,13.0000,7.0000
2,2022-09-01 02:00:00,16.0000,1.0000,20.0000,44.0000,24.0000,16.0000,12.0000
3,2022-09-01 03:00:00,16.0000,1.0000,20.0000,44.0000,22.0000,16.0000,12.0000
4,2022-09-01 04:00:00,17.0000,1.0000,20.0000,50.0000,24.0000,13.0000,8.0000


Sample: poi.csv — first eight columns only


,primary_types,longitude,latitude
0,lifestyle services,114.5766,22.4949
1,lifestyle services,114.5796,22.4922
2,lifestyle services,114.5659,22.5005
3,lifestyle services,114.5774,22.4925
4,lifestyle services,114.5747,22.4946


Weather-header documentation:
Weather Data Table Header：

T：Air temperature (degree Celsius) at 2 metre height above the earth's surfuce.
P0: Atmospheric pressure at weather station level (millimeters of mercury).
P: Atmospheric pressure reduced to mean sea level (millimeters of mercury).
U: Relative humidty (%) at a height of 2 metres above the earth's surface.
nRAIN: 0-normal,1-light rain, 2-moderate rain, 3-heavy rain.
Td: Dewpoint temperature at a height of 2 metres above the earth's surface (degrees Celsius).

Saved: outputs/tables/urbanev_schema_audit.csv


## ۳-۲. تعیین واحد مکانی تقاضا در UrbanEV

در این مرحله بررسی می‌شود که آیا ستون‌های زمانی `duration.csv` و `occupancy.csv` به شناسهٔ ایستگاه‌ها تعلق دارند یا به نواحی ترافیکی (TAZ).

نتیجهٔ این بررسی، سطح مکانی مشترک برای بازطراحی مجموعهٔ آموزشی CHARGED و اعتبارسنجی خارجی را تعیین خواهد کرد.

In [3]:
# Prompt: Verify whether UrbanEV demand time-series columns represent stations or traffic analysis zones.

urbanev_station_info = pd.read_csv(
    URBANEV_RAW_DIR / "inf.csv"
).copy()

duration_header = pd.read_csv(
    URBANEV_RAW_DIR / "duration.csv",
    nrows=0,
)

occupancy_header = pd.read_csv(
    URBANEV_RAW_DIR / "occupancy.csv",
    nrows=0,
)

duration_unit_ids = (
    duration_header.columns[1:]
    .astype(str)
    .tolist()
)

occupancy_unit_ids = (
    occupancy_header.columns[1:]
    .astype(str)
    .tolist()
)

urbanev_station_info["station_id"] = (
    urbanev_station_info["station_id"].astype(str)
)

urbanev_station_info["TAZID"] = (
    urbanev_station_info["TAZID"].astype(str)
)

station_id_set = set(urbanev_station_info["station_id"])
taz_id_set = set(urbanev_station_info["TAZID"])

granularity_audit = pd.DataFrame(
    [
        {
            "station_records": len(urbanev_station_info),
            "unique_station_ids": urbanev_station_info[
                "station_id"
            ].nunique(),
            "unique_taz_ids": urbanev_station_info["TAZID"].nunique(),
            "duration_columns": len(duration_unit_ids),
            "occupancy_columns": len(occupancy_unit_ids),
            "duration_columns_matching_station_ids": len(
                set(duration_unit_ids).intersection(station_id_set)
            ),
            "duration_columns_matching_taz_ids": len(
                set(duration_unit_ids).intersection(taz_id_set)
            ),
            "occupancy_columns_matching_station_ids": len(
                set(occupancy_unit_ids).intersection(station_id_set)
            ),
            "occupancy_columns_matching_taz_ids": len(
                set(occupancy_unit_ids).intersection(taz_id_set)
            ),
        }
    ]
)

taz_station_summary = (
    urbanev_station_info
    .groupby("TAZID", as_index=False)
    .agg(
        station_count=("station_id", "nunique"),
        total_chargers=("charge_count", "sum"),
        area=("area", "first"),
        perimeter=("perimeter", "first"),
    )
)

display(granularity_audit)

print("Station-count distribution across TAZs:")
display(
    taz_station_summary["station_count"]
    .describe()
    .to_frame(name="station_count")
)

granularity_audit.to_csv(
    OUTPUT_TABLES_DIR / "urbanev_spatial_granularity_audit.csv",
    index=False,
    encoding="utf-8-sig",
)

taz_station_summary.to_csv(
    OUTPUT_TABLES_DIR / "urbanev_taz_station_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Saved UrbanEV spatial-granularity audit tables in outputs/tables/")

,station_records,unique_station_ids,unique_taz_ids,duration_columns,occupancy_columns,duration_columns_matching_station_ids,duration_columns_matching_taz_ids,occupancy_columns_matching_station_ids,occupancy_columns_matching_taz_ids
0,1362,1362,275,275,275,56,275,56,275


Station-count distribution across TAZs:


,station_count
count,275.0000
mean,4.9527
std,4.4527
min,1.0000
25%,2.0000
50%,4.0000
75%,7.0000
max,41.0000


Saved UrbanEV spatial-granularity audit tables in outputs/tables/


## ۳-۳. ممیزی زمانی و آماری متغیرهای تقاضا در UrbanEV

در UrbanEV، متغیرهای `duration` و `occupancy` در سطح TAZ ثبت شده‌اند.
در این مرحله فقط کیفیت، پوشش زمانی و مقدارهای نامعتبر آن‌ها بررسی می‌شود؛
هیچ مدلی آموزش داده نمی‌شود و هیچ داده‌ای با CHARGED ادغام نمی‌گردد.

# Prompt: Audit UrbanEV TAZ-level duration and occupancy time series before external validation.

urbanev_duration = pd.read_csv(
    URBANEV_RAW_DIR / "duration.csv"
).copy()

urbanev_occupancy = pd.read_csv(
    URBANEV_RAW_DIR / "occupancy.csv"
).copy()

for frame in [urbanev_duration, urbanev_occupancy]:
    frame["time"] = pd.to_datetime(
        frame["time"],
        errors="coerce",
    )

duration_values = urbanev_duration.drop(
    columns="time"
).apply(pd.to_numeric, errors="coerce")

occupancy_values = urbanev_occupancy.drop(
    columns="time"
).apply(pd.to_numeric, errors="coerce")

def build_time_series_audit(
    frame,
    values,
    variable_name,
):
    sorted_time = frame["time"].sort_values()
    time_gaps = sorted_time.diff().dropna()

    return {
        "variable": variable_name,
        "time_records": len(frame),
        "start_timestamp": frame["time"].min(),
        "end_timestamp": frame["time"].max(),
        "invalid_timestamps": int(frame["time"].isna().sum()),
        "duplicate_timestamps": int(
            frame["time"].duplicated().sum()
        ),
        "most_common_interval": (
            time_gaps.mode().iloc[0]
            if not time_gaps.empty
            else pd.NaT
        ),
        "irregular_time_gaps": int(
            (
                time_gaps
                != time_gaps.mode().iloc[0]
            ).sum()
        )
        if not time_gaps.empty
        else 0,
        "spatial_unit_columns": values.shape[1],
        "missing_values": int(values.isna().sum().sum()),
        "zero_values": int((values == 0).sum().sum()),
        "zero_value_percent": round(
            100 * (values == 0).sum().sum()
            / values.size,
            4,
        ),
        "negative_values": int((values < 0).sum().sum()),
        "minimum_value": values.min().min(),
        "maximum_value": values.max().max(),
        "mean_value": values.stack().mean(),
    }

urbanev_target_audit = pd.DataFrame(
    [
        build_time_series_audit(
            urbanev_duration,
            duration_values,
            "duration",
        ),
        build_time_series_audit(
            urbanev_occupancy,
            occupancy_values,
            "occupancy",
        ),
    ]
)

display(urbanev_target_audit)

urbanev_target_audit.to_csv(
    OUTPUT_TABLES_DIR / "urbanev_taz_target_quality_audit.csv",
    index=False,
    encoding="utf-8-sig",
)

print(
    "Saved:",
    "outputs/tables/urbanev_taz_target_quality_audit.csv",
)

## ۳-۴. ساخت متغیرهای هدف روزانه در سطح TAZ

متغیر اصلی اعتبارسنجی خارجی، مدت شارژ روزانه به‌ازای هر شارژر است.
متغیر اشغال روزانه به‌ازای هر شارژر نیز به‌عنوان تحلیل تکمیلی ذخیره می‌شود.

واحد «duration» مطابق مقدار ثبت‌شده در دیتاست نگه داشته می‌شود؛
تا پیش از تأیید مستندات منبع، آن را ساعت یا دقیقه نام‌گذاری نمی‌کنیم.

In [8]:
# Prompt: Define the project directory used to save intermediate UrbanEV data.

INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

In [9]:
# Prompt: Construct daily TAZ-level charging-demand targets normalized by charging capacity.

urbanev_taz_capacity = (
    urbanev_station_info
    .groupby("TAZID", as_index=False)
    .agg(
        station_count=("station_id", "nunique"),
        total_chargers=("charge_count", "sum"),
        longitude=("longitude", "mean"),
        latitude=("latitude", "mean"),
        area=("area", "first"),
        perimeter=("perimeter", "first"),
    )
    .rename(columns={"TAZID": "taz_id"})
)

urbanev_duration_long = (
    urbanev_duration
    .melt(
        id_vars="time",
        var_name="taz_id",
        value_name="hourly_duration",
    )
)

urbanev_occupancy_long = (
    urbanev_occupancy
    .melt(
        id_vars="time",
        var_name="taz_id",
        value_name="hourly_occupancy",
    )
)

urbanev_duration_long["taz_id"] = (
    urbanev_duration_long["taz_id"].astype(str)
)

urbanev_occupancy_long["taz_id"] = (
    urbanev_occupancy_long["taz_id"].astype(str)
)

urbanev_taz_hourly = (
    urbanev_duration_long
    .merge(
        urbanev_occupancy_long,
        on=["time", "taz_id"],
        how="inner",
        validate="one_to_one",
    )
    .merge(
        urbanev_taz_capacity,
        on="taz_id",
        how="left",
        validate="many_to_one",
    )
)

urbanev_taz_hourly["date"] = (
    urbanev_taz_hourly["time"].dt.normalize()
)

urbanev_taz_daily_targets = (
    urbanev_taz_hourly
    .groupby(
        ["taz_id", "date"],
        as_index=False,
    )
    .agg(
        daily_duration=("hourly_duration", "sum"),
        daily_occupancy=("hourly_occupancy", "sum"),
        total_chargers=("total_chargers", "first"),
        station_count=("station_count", "first"),
        longitude=("longitude", "first"),
        latitude=("latitude", "first"),
        area=("area", "first"),
        perimeter=("perimeter", "first"),
    )
)

urbanev_taz_daily_targets[
    "daily_duration_per_charger"
] = (
    urbanev_taz_daily_targets["daily_duration"]
    / urbanev_taz_daily_targets["total_chargers"]
)

urbanev_taz_daily_targets[
    "daily_occupancy_per_charger"
] = (
    urbanev_taz_daily_targets["daily_occupancy"]
    / urbanev_taz_daily_targets["total_chargers"]
)

urbanev_daily_target_audit = (
    urbanev_taz_daily_targets
    .agg(
        taz_day_records=("taz_id", "size"),
        unique_tazs=("taz_id", "nunique"),
        unique_dates=("date", "nunique"),
        mean_duration_per_charger=(
            "daily_duration_per_charger",
            "mean",
        ),
        median_duration_per_charger=(
            "daily_duration_per_charger",
            "median",
        ),
        zero_duration_percent=(
            "daily_duration_per_charger",
            lambda x: 100 * (x == 0).mean(),
        ),
        mean_occupancy_per_charger=(
            "daily_occupancy_per_charger",
            "mean",
        ),
    )
)

display(urbanev_daily_target_audit)

urbanev_taz_daily_targets.to_csv(
    INTERIM_DIR / "urbanev_taz_day_targets.csv.gz",
    index=False,
    compression="gzip",
)

urbanev_daily_target_audit.to_csv(
    OUTPUT_TABLES_DIR
    / "urbanev_taz_day_target_construction_audit.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Shape:", urbanev_taz_daily_targets.shape)
print(
    "Saved:",
    "data/interim/urbanev_taz_day_targets.csv.gz",
)



,taz_id,date,daily_duration_per_charger,daily_occupancy_per_charger
taz_day_records,"49,775.0000",NaN,NaN,NaN
unique_tazs,275.0000,NaN,NaN,NaN
unique_dates,NaN,181.0000,NaN,NaN
mean_duration_per_charger,NaN,NaN,4.5512,NaN
median_duration_per_charger,NaN,NaN,4.1019,NaN
zero_duration_percent,NaN,NaN,1.8101,NaN
mean_occupancy_per_charger,NaN,NaN,NaN,6.5029


Shape: (49775, 12)
Saved: data/interim/urbanev_taz_day_targets.csv.gz


## ۳-۵. ممیزی و مقایسهٔ دو منبع آب‌وهوایی UrbanEV

UrbanEV دو فایل آب‌وهوا دارد: ایستگاه مرکزی و ایستگاه فرودگاه.
پیش از انتخاب منبع، پوشش زمانی، دادهٔ گمشده و اختلاف متغیرهای مشترک آن‌ها بررسی می‌شود.

In [10]:
# Prompt: Audit and compare UrbanEV central and airport weather records.

weather_sources = {
    "central": pd.read_csv(
        URBANEV_RAW_DIR / "weather_central.csv"
    ),
    "airport": pd.read_csv(
        URBANEV_RAW_DIR / "weather_airport.csv"
    ),
}

weather_audit_rows = []

for source_name, weather_frame in weather_sources.items():
    weather_frame["time"] = pd.to_datetime(
        weather_frame["time"],
        errors="coerce",
    )

    weather_sources[source_name] = weather_frame

    sorted_time = weather_frame["time"].sort_values()
    time_gaps = sorted_time.diff().dropna()
    common_interval = (
        time_gaps.mode().iloc[0]
        if not time_gaps.empty
        else pd.NaT
    )

    weather_audit_rows.append(
        {
            "weather_source": source_name,
            "records": len(weather_frame),
            "start_timestamp": weather_frame["time"].min(),
            "end_timestamp": weather_frame["time"].max(),
            "invalid_timestamps": int(
                weather_frame["time"].isna().sum()
            ),
            "duplicate_timestamps": int(
                weather_frame["time"].duplicated().sum()
            ),
            "most_common_interval": common_interval,
            "irregular_time_gaps": int(
                (time_gaps != common_interval).sum()
            ),
            "missing_values": int(
                weather_frame.drop(
                    columns="time"
                ).isna().sum().sum()
            ),
            "rainy_hour_percent": round(
                100
                * (
                    pd.to_numeric(
                        weather_frame["nRAIN"],
                        errors="coerce",
                    )
                    > 0
                ).mean(),
                4,
            ),
            "mean_temperature": round(
                pd.to_numeric(
                    weather_frame["T"],
                    errors="coerce",
                ).mean(),
                4,
            ),
            "mean_humidity": round(
                pd.to_numeric(
                    weather_frame["U"],
                    errors="coerce",
                ).mean(),
                4,
            ),
        }
    )

urbanev_weather_audit = pd.DataFrame(
    weather_audit_rows
)

weather_comparison = (
    weather_sources["central"]
    .merge(
        weather_sources["airport"],
        on="time",
        how="inner",
        suffixes=("_central", "_airport"),
        validate="one_to_one",
    )
)

weather_difference_audit = pd.DataFrame(
    [
        {
            "matched_hourly_records": len(
                weather_comparison
            ),
            "temperature_mean_absolute_difference": round(
                (
                    weather_comparison["T_central"]
                    - weather_comparison["T_airport"]
                ).abs().mean(),
                4,
            ),
            "humidity_mean_absolute_difference": round(
                (
                    weather_comparison["U_central"]
                    - weather_comparison["U_airport"]
                ).abs().mean(),
                4,
            ),
            "rain_category_agreement_percent": round(
                100
                * (
                    weather_comparison["nRAIN_central"]
                    == weather_comparison["nRAIN_airport"]
                ).mean(),
                4,
            ),
        }
    ]
)

display(urbanev_weather_audit)
display(weather_difference_audit)

urbanev_weather_audit.to_csv(
    OUTPUT_TABLES_DIR
    / "urbanev_weather_source_quality_audit.csv",
    index=False,
    encoding="utf-8-sig",
)

weather_difference_audit.to_csv(
    OUTPUT_TABLES_DIR
    / "urbanev_weather_source_comparison.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Saved weather audit tables in outputs/tables/")

,weather_source,records,start_timestamp,end_timestamp,invalid_timestamps,duplicate_timestamps,most_common_interval,irregular_time_gaps,missing_values,rainy_hour_percent,mean_temperature,mean_humidity
0,central,4344,2022-09-01,2023-02-28 23:00:00,0,0,0 days 01:00:00,0,0,8.5635,21.1766,69.3333
1,airport,4344,2022-09-01,2023-02-28 23:00:00,0,0,0 days 01:00:00,0,0,6.6989,21.8193,65.9735


,matched_hourly_records,temperature_mean_absolute_difference,humidity_mean_absolute_difference,rain_category_agreement_percent
0,4344,1.1052,6.4680,90.5617


Saved weather audit tables in outputs/tables/


## ۳-۶. ساخت ویژگی‌های روزانهٔ زمانی و آب‌وهوایی UrbanEV

منبع اصلی آب‌وهوا، ایستگاه مرکزی است.
ویژگی‌ها شامل تقویم، دمای میانگین، رطوبت میانگین، فشار، دمای نقطهٔ شبنم
و شاخص بارش روزانه هستند. دادهٔ فرودگاه برای تحلیل حساسیت بعدی حفظ می‌شود.

In [11]:
# Prompt: Create leakage-free daily calendar and central-weather features for UrbanEV TAZ targets.

urbanev_weather_daily = (
    weather_sources["central"]
    .assign(date=lambda x: x["time"].dt.normalize())
    .groupby("date", as_index=False)
    .agg(
        temp_mean=("T", "mean"),
        humidity_mean=("U", "mean"),
        station_pressure_mean=("P0", "mean"),
        sea_level_pressure_mean=("P", "mean"),
        dew_point_mean=("Td", "mean"),
        rain_category_mean=("nRAIN", "mean"),
        rainy_hours=(
            "nRAIN",
            lambda x: (pd.to_numeric(x) > 0).sum(),
        ),
    )
)

urbanev_weather_daily["has_precipitation"] = (
    urbanev_weather_daily["rainy_hours"] > 0
).astype(int)

urbanev_weather_daily["recorded_month"] = (
    urbanev_weather_daily["date"].dt.month
)

urbanev_weather_daily["recorded_day_of_week"] = (
    urbanev_weather_daily["date"].dt.dayofweek
)

urbanev_weather_daily["recorded_day_of_year"] = (
    urbanev_weather_daily["date"].dt.dayofyear
)

urbanev_weather_daily["month_sin"] = np.sin(
    2 * np.pi
    * urbanev_weather_daily["recorded_month"]
    / 12
)

urbanev_weather_daily["month_cos"] = np.cos(
    2 * np.pi
    * urbanev_weather_daily["recorded_month"]
    / 12
)

urbanev_taz_day_temporal_features = (
    urbanev_taz_daily_targets
    .merge(
        urbanev_weather_daily,
        on="date",
        how="left",
        validate="many_to_one",
    )
)

temporal_feature_audit = pd.DataFrame(
    {
        "taz_day_records": [
            len(urbanev_taz_day_temporal_features)
        ],
        "unique_tazs": [
            urbanev_taz_day_temporal_features[
                "taz_id"
            ].nunique()
        ],
        "unique_dates": [
            urbanev_taz_day_temporal_features[
                "date"
            ].nunique()
        ],
        "missing_weather_feature_values": [
            int(
                urbanev_taz_day_temporal_features[
                    urbanev_weather_daily.columns.drop("date")
                ]
                .isna()
                .sum()
                .sum()
            )
        ],
        "precipitation_days": [
            int(
                urbanev_weather_daily[
                    "has_precipitation"
                ].sum()
            )
        ],
        "mean_temperature": [
            round(
                urbanev_weather_daily["temp_mean"].mean(),
                4,
            )
        ],
    }
)

display(temporal_feature_audit)

urbanev_taz_day_temporal_features.to_csv(
    INTERIM_DIR
    / "urbanev_taz_day_temporal_features.csv.gz",
    index=False,
    compression="gzip",
)

temporal_feature_audit.to_csv(
    OUTPUT_TABLES_DIR
    / "urbanev_taz_day_temporal_feature_audit.csv",
    index=False,
    encoding="utf-8-sig",
)

print(
    "Shape:",
    urbanev_taz_day_temporal_features.shape,
)
print(
    "Saved:",
    "data/interim/urbanev_taz_day_temporal_features.csv.gz",
)

,taz_day_records,unique_tazs,unique_dates,missing_weather_feature_values,precipitation_days,mean_temperature
0,49775,275,181,0,47,21.1766


Shape: (49775, 25)
Saved: data/interim/urbanev_taz_day_temporal_features.csv.gz


## ۳-۷. ممیزی POIها و سازگاری مختصات مکانی

پیش از ساخت ویژگی‌های مکانی، تعداد، دسته‌ها، داده‌های گمشده و گسترهٔ
مختصات POIها با مختصات ایستگاه‌های شارژ مقایسه می‌شود.
اگر دستگاه مختصات ناسازگار باشد، هیچ اتصال مکانی انجام نخواهد شد.

In [12]:
# Prompt: Audit UrbanEV POIs and compare their spatial extent with charging-station coordinates.

urbanev_poi = pd.read_csv(
    URBANEV_RAW_DIR / "poi.csv"
).copy()

urbanev_poi["longitude"] = pd.to_numeric(
    urbanev_poi["longitude"],
    errors="coerce",
)

urbanev_poi["latitude"] = pd.to_numeric(
    urbanev_poi["latitude"],
    errors="coerce",
)

station_longitude_min = urbanev_station_info[
    "longitude"
].min()

station_longitude_max = urbanev_station_info[
    "longitude"
].max()

station_latitude_min = urbanev_station_info[
    "latitude"
].min()

station_latitude_max = urbanev_station_info[
    "latitude"
].max()

poi_audit = pd.DataFrame(
    {
        "poi_records": [len(urbanev_poi)],
        "poi_type_count": [
            urbanev_poi["primary_types"].nunique()
        ],
        "missing_poi_type": [
            int(urbanev_poi["primary_types"].isna().sum())
        ],
        "missing_poi_coordinates": [
            int(
                urbanev_poi[
                    ["longitude", "latitude"]
                ].isna().any(axis=1).sum()
            )
        ],
        "invalid_poi_coordinates": [
            int(
                (
                    (urbanev_poi["longitude"] < -180)
                    | (urbanev_poi["longitude"] > 180)
                    | (urbanev_poi["latitude"] < -90)
                    | (urbanev_poi["latitude"] > 90)
                ).sum()
            )
        ],
        "poi_longitude_min": [
            urbanev_poi["longitude"].min()
        ],
        "poi_longitude_max": [
            urbanev_poi["longitude"].max()
        ],
        "poi_latitude_min": [
            urbanev_poi["latitude"].min()
        ],
        "poi_latitude_max": [
            urbanev_poi["latitude"].max()
        ],
        "station_longitude_min": [
            station_longitude_min
        ],
        "station_longitude_max": [
            station_longitude_max
        ],
        "station_latitude_min": [
            station_latitude_min
        ],
        "station_latitude_max": [
            station_latitude_max
        ],
    }
)

poi_type_counts = (
    urbanev_poi["primary_types"]
    .fillna("missing")
    .value_counts()
    .rename_axis("poi_type")
    .reset_index(name="poi_count")
)

display(poi_audit)
print("Twenty most frequent POI types:")
display(poi_type_counts.head(20))

poi_audit.to_csv(
    OUTPUT_TABLES_DIR / "urbanev_poi_quality_audit.csv",
    index=False,
    encoding="utf-8-sig",
)

poi_type_counts.to_csv(
    OUTPUT_TABLES_DIR / "urbanev_poi_type_counts.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Saved UrbanEV POI audit tables in outputs/tables/")

,poi_records,poi_type_count,missing_poi_type,missing_poi_coordinates,invalid_poi_coordinates,poi_longitude_min,poi_longitude_max,poi_latitude_min,poi_latitude_max,station_longitude_min,station_longitude_max,station_latitude_min,station_latitude_max
0,712135,3,0,0,0,113.7564,114.6198,22.4086,22.8551,113.7847,114.4935,22.4656,22.8189


Twenty most frequent POI types:


,poi_type,poi_count
0,lifestyle services,393428
1,business and residential,183383
2,food and beverage services,135324


Saved UrbanEV POI audit tables in outputs/tables/


## ۳-۸. کنترل نهایی سازگاری ظرفیت و متغیر هدف در سطح TAZ

در این مرحله کامل‌بودن اتصال بین TAZها، ظرفیت شارژ و هدف روزانه بررسی می‌شود.
این کنترل تضمین می‌کند که نرمال‌سازی تقاضا بر اساس تعداد شارژر برای همهٔ نواحی معتبر است.

In [13]:
# Prompt: Verify TAZ capacity coverage and daily-target consistency before external-model preparation.

urbanev_taz_consistency = (
    urbanev_taz_daily_targets
    .groupby("taz_id", as_index=False)
    .agg(
        observed_days=("date", "nunique"),
        total_chargers=("total_chargers", "first"),
        station_count=("station_count", "first"),
        missing_capacity_values=(
            "total_chargers",
            lambda x: int(x.isna().sum()),
        ),
        nonpositive_capacity_values=(
            "total_chargers",
            lambda x: int((x <= 0).sum()),
        ),
        mean_daily_duration_per_charger=(
            "daily_duration_per_charger",
            "mean",
        ),
        mean_daily_occupancy_per_charger=(
            "daily_occupancy_per_charger",
            "mean",
        ),
    )
)

urbanev_taz_consistency_audit = pd.DataFrame(
    {
        "expected_tazs": [len(urbanev_taz_capacity)],
        "tazs_in_daily_targets": [
            urbanev_taz_consistency["taz_id"].nunique()
        ],
        "tazs_with_181_observed_days": [
            int(
                (
                    urbanev_taz_consistency["observed_days"]
                    == 181
                ).sum()
            )
        ],
        "tazs_with_missing_capacity": [
            int(
                (
                    urbanev_taz_consistency[
                        "missing_capacity_values"
                    ]
                    > 0
                ).sum()
            )
        ],
        "tazs_with_nonpositive_capacity": [
            int(
                (
                    urbanev_taz_consistency[
                        "nonpositive_capacity_values"
                    ]
                    > 0
                ).sum()
            )
        ],
        "minimum_total_chargers": [
            urbanev_taz_consistency[
                "total_chargers"
            ].min()
        ],
        "maximum_total_chargers": [
            urbanev_taz_consistency[
                "total_chargers"
            ].max()
        ],
    }
)

display(urbanev_taz_consistency_audit)

print("Distribution of charging capacity across TAZs:")
display(
    urbanev_taz_consistency[
        ["station_count", "total_chargers"]
    ].describe()
)

urbanev_taz_consistency.to_csv(
    OUTPUT_TABLES_DIR
    / "urbanev_taz_capacity_target_consistency.csv",
    index=False,
    encoding="utf-8-sig",
)

urbanev_taz_consistency_audit.to_csv(
    OUTPUT_TABLES_DIR
    / "urbanev_taz_capacity_target_consistency_audit.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Saved final UrbanEV consistency tables in outputs/tables/")

,expected_tazs,tazs_in_daily_targets,tazs_with_181_observed_days,tazs_with_missing_capacity,tazs_with_nonpositive_capacity,minimum_total_chargers,maximum_total_chargers
0,275,275,275,0,0,4,373


Distribution of charging capacity across TAZs:


,station_count,total_chargers
count,275.0000,275.0000
mean,4.9527,63.7527
std,4.4527,56.7336
min,1.0000,4.0000
25%,2.0000,24.0000
50%,4.0000,48.0000
75%,7.0000,84.0000
max,41.0000,373.0000


Saved final UrbanEV consistency tables in outputs/tables/
